# pinn_bench v2 -- GPU sweep (NLS benchmark + ConflictAverseAdam optimizer)

This notebook runs the **v2 extension** of the pinn_bench benchmark suite on a Colab GPU:
a 5th benchmark problem (the nonlinear Schrodinger equation), a 7th optimizer
(`ConflictAverseAdam`, a new conflict-resolving optimizer motivated by this project's own
gradient-conflict finding), and 10 seeds instead of 5 for tighter statistics.

**Instructions**
1. Runtime -> Change runtime type -> select a GPU (T4 is fine).
2. Run the cells in order, top to bottom.
3. When Cell 2 asks you to upload a file, upload `pinn_bench_v2_repo.zip` (provided alongside
   this notebook).
4. First run keeps `QUICK_MODE = True` to confirm everything works end to end (~1-2 minutes).
5. Once that succeeds, set `QUICK_MODE = False` in Cell 4 and re-run from there for the real
   350-run sweep (`configs/full_v2.yaml`). Wall-clock time depends on the assigned GPU --
   watch the first few "done in Xs" lines to estimate the total before committing to the full
   run.
6. At the end, `pinn_bench_v2_results.zip` is created and auto-downloaded.


In [ ]:
# STEP 1: Confirm a GPU is attached
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("No GPU detected -- go to Runtime -> Change runtime type -> GPU, then re-run this cell.")


In [ ]:
# STEP 2: Upload the pinn_bench_v2_repo.zip file provided alongside this notebook
from google.colab import files
uploaded = files.upload()
print("Uploaded:", list(uploaded.keys()))


In [ ]:
# STEP 3: Unzip and install the package (editable install, so configs/scripts are used in place)
import os, zipfile

zip_name = [f for f in uploaded.keys() if f.endswith(".zip")][0]
with zipfile.ZipFile(zip_name, "r") as zf:
    zf.extractall("pinn_bench_src")

# Handle both "zip contains a single top-level folder" and "zip contains files at root"
root = "pinn_bench_src"
entries = os.listdir(root)
if len(entries) == 1 and os.path.isdir(os.path.join(root, entries[0])):
    root = os.path.join(root, entries[0])
print("Package root:", root)

!pip install -q -e "{root}"
!pip install -q scipy pandas matplotlib pyyaml


In [ ]:
# STEP 4: Configuration -- QUICK_MODE=True first, then False for the real sweep
QUICK_MODE = True

config_path = f"{root}/configs/quick_v2.yaml" if QUICK_MODE else f"{root}/configs/full_v2.yaml"
print("Using config:", config_path)
!cat "{config_path}"


In [ ]:
# STEP 5: Run the sweep (streams progress live; this is the long-running cell)
import os
os.chdir(root)
!python -m pinn_bench.run --config "configs/{'quick_v2' if QUICK_MODE else 'full_v2'}.yaml" --device cuda


## Optional: hyperparameter tuning and matched-loss ablation

These two cells are optional, additional rigor upgrades -- run them only after you are happy
with the main sweep above (or in parallel in a separate Colab session, since they're
independent of it). Both are slower small sweeps of their own; expect each to take a
noticeable fraction of the main sweep's time.


In [ ]:
# OPTIONAL: per-problem learning-rate tuning (writes configs/full_v2_tuned.yaml)
# Uncomment to run:
# !python scripts/tune_hyperparameters.py --base-config configs/full_v2.yaml \
#     --out configs/full_v2_tuned.yaml --device cuda


In [ ]:
# OPTIONAL: matched-loss-value gradient-conflict ablation (addresses the v1
# under-training confound directly -- see scripts/matched_loss_ablation.py's docstring)
# Uncomment to run:
# !python scripts/matched_loss_ablation.py \
#     --problems heat burgers allen_cahn kdv nls \
#     --optimizers Adam Shampoo SOAP Muon ConflictAverseAdam \
#     --seeds 0 1 2 --max-steps 3000 --device cuda \
#     --out results/matched_loss_ablation.csv


In [ ]:
# STEP 6: Zip the results directory and download it
import shutil
from google.colab import files as colab_files

results_dir = f"results/{'quick_v2' if QUICK_MODE else 'full_v2'}"
zip_base = "pinn_bench_v2_results"
shutil.make_archive(zip_base, "zip", root_dir=results_dir)
print("Created:", zip_base + ".zip")
colab_files.download(zip_base + ".zip")
